# Pipeline Evaluation Notebook

Living notebook that tracks how each Stage of the pipeline performs against
the frozen baseline. Re-run after every Stage and append a new section. The
cross-stage comparison at the bottom shows whether the new work moved the
needle.

**No Docker / no Ollama / no Elasticsearch are required for the Stage 0
baseline cells below.** `thesis_final_benchmark.csv` already contains the
frozen Tier-1 (`model_label`) and Tier-2 (`agent_final_decision`) verdicts
from the internship benchmark, plus the manually-labelled
`human_ground_truth`. Everything in §1 runs purely offline.

Live-infra requirements only appear when a future Stage needs to *generate*
new benchmark data — those cells will carry a `[LIVE INFRA]` tag in their
header.

## How to extend this notebook

1. When a Stage lands, append a new top-level section
   (e.g. `## 4. Stage 2 — Temporal Memory`).
2. Reuse the helpers defined in §0
   (`load_benchmark`, `agent_effective_label`, `compute_three_class_accuracy`, …).
3. Add the resulting metrics dict to `RESULTS_REGISTRY` near the end.
4. The cross-stage comparison table picks it up automatically.

Related docs: [`docs/ROADMAP.md`](../docs/ROADMAP.md),
[`docs/EVAL.md`](../docs/EVAL.md), [`docs/PROMPTS.md`](../docs/PROMPTS.md).

## 0. Setup

In [1]:
import os
from collections import OrderedDict

import numpy as np
import pandas as pd


# Path resolution — works whether the notebook is opened from the repo root
# or from notebooks/.
def _find_benchmark():
    for candidate in ("thesis_final_benchmark.csv", "../thesis_final_benchmark.csv"):
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(
        "Could not find thesis_final_benchmark.csv — run this notebook from the "
        "repo root or from notebooks/."
    )


BENCHMARK_CSV = _find_benchmark()
print(f"Benchmark: {BENCHMARK_CSV}")

# Closed taxonomy (mirrors src/shared_utils/prompts.py::ENV_DOMAINS).
ENV_DOMAINS = [
    "Gaming", "Politics", "News", "Music", "Sports",
    "Education", "Entertainment", "Technology", "Lifestyle", "General",
]

# Tier-1 label space.
LABEL_CLASSES = ["NORMAL", "OFFENSIVE", "HATE"]

Benchmark: ../thesis_final_benchmark.csv


In [2]:
def load_benchmark(path=BENCHMARK_CSV):
    """Load + normalise the benchmark CSV. Returns a clean DataFrame."""
    df = pd.read_csv(path)
    df["human_ground_truth"] = (
        df["human_ground_truth"].fillna("NORMAL").astype(str).str.strip().str.upper()
    )
    df["model_label"] = df["model_label"].astype(str).str.strip().str.upper()
    df["agent_final_decision"] = (
        df["agent_final_decision"].astype(str).str.strip().str.lower()
    )
    df["env_domain"] = df["env_domain"].fillna("Unknown").astype(str)
    return df


def main_domain(raw):
    """Collapse the legacy open-taxonomy 'Gaming/Survival' to 'Gaming'."""
    if pd.isna(raw):
        return "Unknown"
    return str(raw).split("/")[0].strip() or "Unknown"


def agent_effective_label(model_label, agent_decision):
    """
    Map (Tier-1 label, Tier-2 verdict) -> effective Tier-2 prediction.

    - 'correct'        -> keep the Tier-1 label
    - 'false positive' -> NORMAL    (Tier-2 cleared the static flag)
    - 'false negative' -> OFFENSIVE (Tier-2 says static missed; conservative
                                     choice — Report Table 4 shows only
                                     2/459 Agent-Hate predictions, so HATE
                                     is rarely the right flip.)
    - anything else (parsing errors) -> keep Tier-1 label
    """
    if agent_decision == "correct":
        return model_label
    if agent_decision == "false positive":
        return "NORMAL"
    if agent_decision == "false negative":
        return "OFFENSIVE"
    return model_label


def compute_three_class_accuracy(predictions, ground_truth):
    return float((predictions == ground_truth).sum()) / len(ground_truth)


def compute_binary_accuracy(predictions, ground_truth, toxic={"HATE", "OFFENSIVE"}):
    """Accuracy when collapsing labels to (normal vs toxic)."""
    pred_b = predictions.isin(toxic)
    gt_b = ground_truth.isin(toxic)
    return float((pred_b == gt_b).sum()) / len(ground_truth)


def confusion_matrix(predictions, ground_truth, classes=LABEL_CLASSES):
    cm = pd.crosstab(
        ground_truth, predictions,
        rownames=["Human"], colnames=["Predicted"], dropna=False,
    )
    return cm.reindex(index=classes, columns=classes, fill_value=0)

## 1. Stage 0 — Baseline: Static vs Agent vs Human

### 1.1 Load benchmark

In [3]:
df = load_benchmark()
print(f"Benchmark size: {len(df)} records")
print(f"Columns: {list(df.columns)}")
df.head(3)

Benchmark size: 459 records
Columns: ['env_domain', 'text', 'model_label', 'agent_final_decision', 'human_ground_truth', 'env_strictness', 'processing_time_ms', 'agent_latency_seconds']


,env_domain,text,model_label,agent_final_decision,human_ground_truth,env_strictness,processing_time_ms,agent_latency_seconds
0,Politics/News,TRYING TO INFLUENCE ANOTHER COUNTRY ELECTIONS ...,HATE,false positive,NORMAL,high,128.22,4.74
1,Gaming,awie would be better off working with the enemy,HATE,false positive,NORMAL,low,80.45,6.91
2,Gaming,"Most adults are tired of that, viewbotz",HATE,false positive,NORMAL,low,73.01,6.67


### 1.2 Class distribution — reproduces Report Table 4

In [4]:
df["agent_pred"] = df.apply(
    lambda r: agent_effective_label(r["model_label"], r["agent_final_decision"]),
    axis=1,
)

table4 = pd.DataFrame({
    "Static Classifier": df["model_label"].value_counts().reindex(LABEL_CLASSES, fill_value=0),
    "Agentic Auditor":   df["agent_pred"].value_counts().reindex(LABEL_CLASSES, fill_value=0),
    "Human Expert":      df["human_ground_truth"].value_counts().reindex(LABEL_CLASSES, fill_value=0),
})
table4

,Static Classifier,Agentic Auditor,Human Expert
NORMAL,347,384,384
OFFENSIVE,70,73,71
HATE,42,2,4


### 1.3 Agent decision breakdown — reproduces Report Table 6

In [5]:
agent_breakdown = df["agent_final_decision"].value_counts()
agent_breakdown.name = "Sample Count"
print(agent_breakdown.to_string())
print(f"\nTotal Evaluated: {len(df)}")

agent_final_decision
correct               312
false positive         91
false negative         54
json parsing error      2

Total Evaluated: 459


### 1.4 Three-way accuracy — reproduces Report Table 8

In [6]:
static_acc = compute_three_class_accuracy(df["model_label"], df["human_ground_truth"])
agent_acc = compute_three_class_accuracy(df["agent_pred"], df["human_ground_truth"])
static_agent_consensus = compute_three_class_accuracy(df["model_label"], df["agent_pred"])
unanimous = (
    (df["model_label"] == df["agent_pred"]) &
    (df["agent_pred"] == df["human_ground_truth"])
).sum() / len(df)
agent_human_static_failed = (
    (df["agent_pred"] == df["human_ground_truth"]) &
    (df["model_label"] != df["human_ground_truth"])
).sum() / len(df)

agreement = pd.Series({
    "Human vs Agent  (Agent Accuracy, 3-class)":  agent_acc,
    "Human vs Static (Static Accuracy, 3-class)": static_acc,
    "Static vs Agent (AI Consensus)":             static_agent_consensus,
    "Unanimous (all 3 match)":                    unanimous,
    "Agent & Human Match (Static Failed)":        agent_human_static_failed,
})
agreement.apply(lambda x: f"{x:.1%}")

Human vs Agent  (Agent Accuracy, 3-class)     93.5%
Human vs Static (Static Accuracy, 3-class)    71.0%
Static vs Agent (AI Consensus)                68.4%
Unanimous (all 3 match)                       67.1%
Agent & Human Match (Static Failed)           26.4%
dtype: object

### 1.5 Per-domain accuracy

Uses `main_domain()` to collapse the legacy open-taxonomy values (`Gaming/Survival` → `Gaming`) so the breakdown matches the closed taxonomy introduced in Stage 0.

In [7]:
df["main_domain"] = df["env_domain"].apply(main_domain)

def _row(g):
    s_acc = compute_three_class_accuracy(g["model_label"], g["human_ground_truth"])
    a_acc = compute_three_class_accuracy(g["agent_pred"], g["human_ground_truth"])
    return pd.Series({
        "n": len(g),
        "Static acc": s_acc,
        "Agent acc":  a_acc,
        "Delta (Agent - Static)": a_acc - s_acc,
    })

per_domain = (
    df.groupby("main_domain")
    .apply(_row)
    .sort_values("n", ascending=False)
)
per_domain

C:\Users\parsa\AppData\Local\Temp\ipykernel_9168\600434415.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_row)


,n,Static acc,Agent acc,Delta (Agent - Static)
main_domain,,,,
Gaming,407.0,0.700246,0.941032,0.240786
News,30.0,0.866667,0.833333,-0.033333
Politics,14.0,0.571429,0.928571,0.357143
Music,8.0,0.875000,1.000000,0.125000


### 1.6 Confusion matrices

In [8]:
print("Static (Tier-1 DistilBERT) vs Human:")
print(confusion_matrix(df["model_label"], df["human_ground_truth"]).to_string())

print("\nAgent (Tier-2) vs Human:")
print(confusion_matrix(df["agent_pred"], df["human_ground_truth"]).to_string())

Static (Tier-1 DistilBERT) vs Human:
Predicted  NORMAL  OFFENSIVE  HATE
Human                             
NORMAL        301         47    36
OFFENSIVE      44         23     4
HATE            2          0     2

Agent (Tier-2) vs Human:
Predicted  NORMAL  OFFENSIVE  HATE
Human                             
NORMAL        370         14     0
OFFENSIVE      14         57     0
HATE            0          2     2


### 1.7 Latency — reproduces Report Table 7

Both numbers are means over the benchmark records. The 192.6× overhead the report cites is the per-record cost the asynchronous Tier-2 sweep incurs — it does not affect the user-facing Tier-1 throughput since Tier-2 runs offline.

In [9]:
t1_ms = df["processing_time_ms"].dropna().mean()
t2_ms = (df["agent_latency_seconds"].dropna() * 1000).mean()

latency = pd.Series({
    "Tier-1 (Static) — mean ms":  f"{t1_ms:.2f} ms",
    "Tier-2 (Agent) — mean ms":   f"{t2_ms:.2f} ms",
    "Overhead (Tier-2 / Tier-1)": f"{t2_ms / t1_ms:.1f}x",
})
print(latency.to_string())

Tier-1 (Static) — mean ms       36.10 ms
Tier-2 (Agent) — mean ms      6951.81 ms
Overhead (Tier-2 / Tier-1)        192.6x


## 2. DistilBERT training reference

Numbers below come from `src/Hate_speach_Project.ipynb` and Internship Report §4.1–4.2. They describe Tier-1 itself — independent of this notebook's CSV — and are kept here as documented constants so the thesis can cite a single source. **Do not** re-run training from this notebook; the GPU run is hours-long and the artefact is already in `models/bert_final/`.

In [10]:
DISTILBERT_TRAINING = {
    "dataset_size_total":         52972,
    "dataset_train_subset":       43000,
    "train_val_split":            "80/20",
    "epochs":                     2,
    "learning_rate":              5e-5,
    "weight_decay":               0.01,
    "warmup_steps":               100,
    "max_token_length":           128,
    "train_batch_size":           8,
    "eval_batch_size":            16,
    "optimizer":                  "AdamW",
    "val_accuracy":               0.7944,
    "val_f1":                     0.796,
    "classical_lr_accuracy":      0.76,
    "classical_lr_hate_f1":       0.66,
    "gpu":                        "NVIDIA GTX 1650 (4GB VRAM)",
}
pd.Series(DISTILBERT_TRAINING)

dataset_size_total                            52972
dataset_train_subset                          43000
train_val_split                               80/20
epochs                                            2
learning_rate                               0.00005
weight_decay                                   0.01
warmup_steps                                    100
max_token_length                                128
train_batch_size                                  8
eval_batch_size                                  16
optimizer                                     AdamW
val_accuracy                                 0.7944
val_f1                                        0.796
classical_lr_accuracy                          0.76
classical_lr_hate_f1                           0.66
gpu                      NVIDIA GTX 1650 (4GB VRAM)
dtype: object

## 3. Stage results registry

Every Stage appends one row here. Stage 0 has two: the static baseline (Tier-1 only) and the hybrid (Tier-1 + Tier-2 internship verdicts). Future stages mutate `RESULTS_REGISTRY` in their own cell, then the cross-stage table at the bottom picks the new row up automatically.

In [11]:
RESULTS_REGISTRY = OrderedDict()

RESULTS_REGISTRY["Stage 0 — Static (Tier-1)"] = {
    "accuracy_3class":  static_acc,
    "accuracy_binary":  compute_binary_accuracy(df["model_label"], df["human_ground_truth"]),
    "mean_latency_ms":  t1_ms,
    "n_records":        len(df),
    "notes":            "DistilBERT alone, no Tier-2 overlay",
}

RESULTS_REGISTRY["Stage 0 — Static + Agent (Tier-1 + Tier-2)"] = {
    "accuracy_3class":  agent_acc,
    "accuracy_binary":  compute_binary_accuracy(df["agent_pred"], df["human_ground_truth"]),
    "mean_latency_ms":  t1_ms + t2_ms,
    "n_records":        len(df),
    "notes":            "Hybrid pipeline, frozen internship verdicts (gpt-oss:120b-cloud)",
}

# ----------------------------------------------------------------------
# Future stages append entries below this line. Example:
#
# RESULTS_REGISTRY["Stage 2 — + Temporal Memory"] = {...}
# RESULTS_REGISTRY["Stage 3 — + RAG"]              = {...}
# ----------------------------------------------------------------------

pd.DataFrame(RESULTS_REGISTRY).T

,accuracy_3class,accuracy_binary,mean_latency_ms,n_records,notes
Stage 0 — Static (Tier-1),0.71024,0.718954,36.098388,459,"DistilBERT alone, no Tier-2 overlay"
Stage 0 — Static + Agent (Tier-1 + Tier-2),0.934641,0.938998,6987.906667,459,"Hybrid pipeline, frozen internship verdicts (g..."


## 4. Template — adding a future Stage

Copy the cell below into a new section when a Stage lands. The comments mark exactly what each subsequent Stage needs to change.

In [12]:
# ----------------------------------------------------------------------
# TEMPLATE — duplicate this cell into its own section for each Stage.
# ----------------------------------------------------------------------
#
# # Stage N — <name>
#
# # 1. Load the Stage-N CSV.
# #    - If the Stage only modifies the agent verdict (prompt/model change),
# #      re-run the live pipeline against the SAME texts and export with:
# #          python src/export_subset.py
# #    - If the Stage adds new data, export a fresh CSV with a stage suffix.
# #
# # [LIVE INFRA] needed only when generating the new CSV; not when reading it.
# stage_n_df = load_benchmark("stage_N_benchmark.csv")
#
# # 2. Compute the agent's effective prediction.
# stage_n_df["agent_pred"] = stage_n_df.apply(
#     lambda r: agent_effective_label(r["model_label"], r["agent_final_decision"]),
#     axis=1,
# )
#
# # 3. Compute metrics.
# acc_3 = compute_three_class_accuracy(stage_n_df["agent_pred"], stage_n_df["human_ground_truth"])
# acc_b = compute_binary_accuracy(stage_n_df["agent_pred"], stage_n_df["human_ground_truth"])
# t1 = stage_n_df["processing_time_ms"].dropna().mean()
# t2 = (stage_n_df["agent_latency_seconds"].dropna() * 1000).mean()
#
# # 4. Register.
# RESULTS_REGISTRY["Stage N — <name>"] = {
#     "accuracy_3class":  acc_3,
#     "accuracy_binary":  acc_b,
#     "mean_latency_ms":  t1 + t2,
#     "n_records":        len(stage_n_df),
#     "notes":            "<what changed since the previous Stage>",
# }
#
# pd.DataFrame(RESULTS_REGISTRY).T

## 5. Cross-stage comparison

Final readout. Re-render after any Stage appends to `RESULTS_REGISTRY`. When a Stage's row appears here, the thesis can cite the delta from the previous row as that Stage's contribution.

In [13]:
comparison = pd.DataFrame(RESULTS_REGISTRY).T.copy()
comparison["accuracy_3class"] = comparison["accuracy_3class"].astype(float).apply(lambda x: f"{x:.1%}")
comparison["accuracy_binary"] = comparison["accuracy_binary"].astype(float).apply(lambda x: f"{x:.1%}")
comparison["mean_latency_ms"] = comparison["mean_latency_ms"].astype(float).apply(lambda x: f"{x:.1f} ms")
comparison

,accuracy_3class,accuracy_binary,mean_latency_ms,n_records,notes
Stage 0 — Static (Tier-1),71.0%,71.9%,36.1 ms,459,"DistilBERT alone, no Tier-2 overlay"
Stage 0 — Static + Agent (Tier-1 + Tier-2),93.5%,93.9%,6987.9 ms,459,"Hybrid pipeline, frozen internship verdicts (g..."
